# Paper 3 — 06 · Aggregate + figures

Reads every `results/<run_tag>__safety.json` + `__capability.json` produced
by notebooks 04 and 05, builds:

1. The headline table (anchor × condition × dim) with Wilson CIs.
2. Safety-vs-k figure (METHOD §9 headline ablation).
3. Alignment-tax scatter: capability_aligned − capability_base.
4. Scaling plot: RD-DPO benefit vs model size on the Qwen-2.5 track.
5. LaTeX snippet `manuscript/tables/results.tex`.

In [ ]:
%%capture
!pip install -U pandas matplotlib seaborn -q

In [ ]:
import os, json, gc, time, hashlib, math, sys
from pathlib import Path
from datetime import datetime
import torch

# --- Drive ---
from google.colab import drive
drive.mount("/content/drive")

# --- Paths ---
DRIVE_ROOT  = Path("/content/drive/MyDrive/PhD/paper3-alignment")
PAPER2_ROOT = Path("/content/drive/MyDrive/PhD/paper2-benchmark")

PROBE_DIR    = DRIVE_ROOT / "data" / "probes"
PREFS_DIR    = DRIVE_ROOT / "data" / "preferences"
SPLITS_DIR   = DRIVE_ROOT / "data" / "splits"
ADAPTERS_DIR = DRIVE_ROOT / "adapters"
RESULTS_DIR  = DRIVE_ROOT / "results"
LOGS_DIR     = DRIVE_ROOT / "logs"
FIG_DIR      = DRIVE_ROOT / "figures"
for d in [PROBE_DIR, PREFS_DIR, SPLITS_DIR, ADAPTERS_DIR, RESULTS_DIR, LOGS_DIR, FIG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# --- Paper 2 reuse: judges, llm_judge, holdout splits ---
sys.path.insert(0, str(PAPER2_ROOT / "src"))

# --- A100 sanity + perf toggles ---
assert torch.cuda.is_available(), "GPU not available -- switch runtime to A100 GPU."
DEVICE_NAME = torch.cuda.get_device_name(0)
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {DEVICE_NAME}   VRAM: {VRAM_GB:.1f} GB   torch={torch.__version__}")
if "A100" not in DEVICE_NAME:
    print(f"WARNING: expected A100, got {DEVICE_NAME}. Re-tune BATCH_SIZE/grad-accum below.")

torch.set_float32_matmul_precision("high")          # TF32 for fp32 matmuls
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True               # autotune for fixed shapes

# --- HF / OpenRouter auth (set in Colab Secrets, not in code) ---
try:
    from google.colab import userdata
    if not os.environ.get("HF_TOKEN"):
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN") or ""
    if os.environ.get("HF_TOKEN"):
        from huggingface_hub import login as _hf_login
        _hf_login(os.environ["HF_TOKEN"], add_to_git_credential=False)
    if not os.environ.get("OPENROUTER_API_KEY"):
        os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY") or ""
except Exception as _e:
    print(f"(secrets not configured: {_e}; set HF_TOKEN / OPENROUTER_API_KEY in Colab secrets)")


def load_kwargs_for(family: str) -> dict:
    """A100-tuned dtype + attention impl per model family.

    Why bf16 everywhere: A100 has bf16 tensor cores at the same throughput as
    fp16, bf16 has the dynamic range of fp32 (no overflow on long sequences),
    and bf16 is the training dtype for all anchor families. Using fp16 here
    silently broke Gemma 3 in Paper 2 (953 empty-string outputs).
    """
    common = dict(torch_dtype=torch.bfloat16, device_map={"": 0})
    if family.startswith("gemma"):
        # Gemma 3 sliding-window attention is brittle with flash-attn-2.
        return {**common, "attn_implementation": "eager"}
    try:
        import flash_attn  # noqa: F401
        return {**common, "attn_implementation": "flash_attention_2"}
    except ImportError:
        return {**common, "attn_implementation": "sdpa"}


def family_of(anchor_id: str) -> str:
    a = anchor_id.lower()
    if "qwen2.5" in a: return "qwen2.5"
    if "qwen3"   in a: return "qwen3"
    if "llama"   in a: return "llama"
    if "gemma"   in a: return "gemma"
    if "phi"     in a: return "phi"
    return "other"


def short_of(anchor_id: str) -> str:
    return (anchor_id.split("/")[-1].lower()
            .replace("-instruct", "").replace("-it", ""))

print("Bootstrap done.")

## Collect results

In [ ]:
import pandas as pd

def parse_run_tag(tag: str):
    short, cond, seed_s = tag.split("__")
    return short, cond, int(seed_s.replace("seed", ""))

rows = []
for path in sorted(RESULTS_DIR.glob("*__safety.json")):
    tag = path.stem.replace("__safety", "")
    short, cond, seed = parse_run_tag(tag)
    d = json.loads(path.read_text())
    for dim, m in d.get("safety", {}).items():
        rows.append({
            "anchor": short, "condition": cond, "seed": seed,
            "dim": dim, "rate": m.get("refusal_rate", 0),
            "n": m.get("n", 0),
            "wilson_lo": m.get("wilson_95", [0, 0])[0],
            "wilson_hi": m.get("wilson_95", [0, 0])[1],
        })
df = pd.DataFrame(rows)
print(df.head())
print(f"\n{len(df)} rows, {df['anchor'].nunique()} anchors, "
      f"{df['condition'].nunique()} conditions")

## Headline table (mean over seeds, per anchor × condition × dim)

In [ ]:
summary = (
    df.groupby(["anchor", "condition", "dim"])
      .agg(rate=("rate", "mean"),
           n=("n", "mean"),
           wilson_lo=("wilson_lo", "min"),
           wilson_hi=("wilson_hi", "max"),
           n_seeds=("seed", "nunique"))
      .reset_index()
)
pivot = summary.pivot_table(
    index=["anchor", "condition"], columns="dim", values="rate",
).round(3)
print(pivot)
pivot.to_csv(RESULTS_DIR / "summary_pivot.csv")
print(f"\nsaved {RESULTS_DIR / 'summary_pivot.csv'}")

## Safety-vs-k figure

In [ ]:
import matplotlib.pyplot as plt

rd = df[df["condition"].str.startswith("rd-dpo-k")]
if not rd.empty:
    rd = rd.copy()
    rd["k"] = rd["condition"].str.extract(r"k(\d+)").astype(int)

    fig, axes = plt.subplots(1, 3, figsize=(12, 3.5), sharey=True)
    for ax, dim in zip(axes, ["toxicity", "jailbreak", "crosslingual"]):
        sub = rd[rd["dim"] == dim].groupby(["anchor", "k"])["rate"].mean().reset_index()
        for anchor, g in sub.groupby("anchor"):
            ax.plot(g["k"], g["rate"], marker="o", label=anchor)
        ax.set_title(f"{dim} refusal vs k")
        ax.set_xlabel("k (probe-selected blocks)")
        ax.set_xscale("log", base=2)
        ax.set_ylim(0, 1.0)
    axes[0].set_ylabel("judged refusal rate")
    axes[0].legend(loc="lower right", fontsize=8)
    plt.tight_layout()
    out_fig = FIG_DIR / "safety_vs_k.pdf"
    plt.savefig(out_fig, bbox_inches="tight")
    plt.savefig(out_fig.with_suffix(".png"), dpi=120, bbox_inches="tight")
    plt.show()
    print(f"saved {out_fig}")
else:
    print("(no rd-dpo-k* runs yet; skipping safety_vs_k figure)")

## Alignment-tax scatter

Per-condition x = capability_aligned − capability_base, y = safety lift over base. Top-left quadrant = capability preserved, safety up.

In [ ]:
cap_rows = []
for path in sorted(RESULTS_DIR.glob("*__capability.json")):
    tag = path.stem.replace("__capability", "")
    short, cond, seed = parse_run_tag(tag)
    cap = json.loads(path.read_text())
    en = cap.get("capability_en", {})
    mmlu   = (en.get("mmlu") or {}).get("acc", 0)
    ro_ppl = cap.get("capability_ro", {}).get("culturax_ppl", float("nan"))
    cap_rows.append({"anchor": short, "condition": cond, "seed": seed,
                     "mmlu": mmlu, "ro_ppl": ro_ppl})

cap_df = pd.DataFrame(cap_rows)
base = cap_df[cap_df["condition"] == "base"].groupby("anchor")[["mmlu", "ro_ppl"]].mean()
delta_rows = []
for _, r in cap_df.iterrows():
    if r["anchor"] in base.index:
        b = base.loc[r["anchor"]]
        delta_rows.append({**r.to_dict(),
                           "delta_mmlu":  r["mmlu"]   - b["mmlu"],
                           "delta_ro_ppl": r["ro_ppl"] - b["ro_ppl"]})
cap_delta = pd.DataFrame(delta_rows)
print(cap_delta.head())
cap_delta.to_csv(RESULTS_DIR / "capability_delta.csv", index=False)
print(f"saved {RESULTS_DIR / 'capability_delta.csv'}")

## LaTeX results-table snippet

In [ ]:
tex_lines = ["% Auto-generated by 06_aggregate_and_figures.ipynb",
             r"\begin{tabular}{llrrrrr}",
             r"\toprule",
             r"Anchor & Condition & Tox.\ RR & JB.\ RR & XL.\ RR & OverRef & Bias \\",
             r"\midrule"]
for (a, c), g in summary.groupby(["anchor", "condition"]):
    cells = {row["dim"]: row["rate"] for _, row in g.iterrows()}
    tex_lines.append(
        f"{a} & {c} & {cells.get('toxicity', 0):.3f} & "
        f"{cells.get('jailbreak', 0):.3f} & {cells.get('crosslingual', 0):.3f} & "
        f"{cells.get('overrefusal', 0):.3f} & {cells.get('bias', 0):.3f} \\\\"
    )
tex_lines += [r"\bottomrule", r"\end{tabular}"]

out_tex = RESULTS_DIR / "results_table.tex"
out_tex.write_text("\n".join(tex_lines))
print(out_tex.read_text())
print(f"\nsaved {out_tex}")